<a href="https://colab.research.google.com/github/Harshit10880/natural_language_processing/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


main things to convert text into numbers

In [4]:
unique_emotion = df['emotion'].unique()
emotion_numbers = {}
i = 0

for emo in unique_emotion:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)


In [5]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


convert all text into lower case

In [6]:
df['text'] = df['text'].apply(lambda x : x.lower())

remove punctuation

In [7]:
import string
def remove_punc(txt):
 return txt.translate(str.maketrans('','',string.punctuation))

In [8]:
df['text'] = df['text'].apply(remove_punc)

remove numbers

In [9]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

remove url and links

remove html tags

remove emojis and special characters

In [10]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

remove stopwords

In [11]:
import nltk

In [12]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [13]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
stop_words = set(stopwords.words('english'))

In [15]:
# stop_words
len(stop_words)

198

*theory kind of in one sentence if there is stop word inside sentence so we have to see that inside list of stop words and that sentence stop words exists in this stop words list so have to remove other if not exist so keep as it is*

In [16]:
def remove(txt):
  # words = txt.tokenize() due to some issuse in colab cant use tokenize instead will use split
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [17]:
df['text'] = df['text'].apply(remove)

In [18]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

countvectorizer for bag of words or like tht

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [23]:
bow_vectorizer = CountVectorizer()

In [26]:
x_train_bow = bow_vectorizer.fit_transform(X_train)
x_test_bow = bow_vectorizer.transform(X_test)

In [25]:
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116059 stored elements and shape (12800, 13361)>

In [28]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [29]:
nb_model = MultinomialNB()
nb_model.fit(x_train_bow, y_train)

MultinomialNB()

In [30]:
pred_nb = nb_model.predict(x_test_bow)

In [31]:
print(accuracy_score(y_test, pred_nb))
print(confusion_matrix(y_test, pred_nb))
print(classification_report(y_test, pred_nb))

0.768125
[[899   8   3   0   4  32]
 [ 87 271   0   0  13  56]
 [ 51   6  79   0   4 156]
 [ 40   0   1   6  17  49]
 [ 87  15   0   0 226  69]
 [ 36   4   2   0   2 977]]
              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.27      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



In [32]:
tfidf_vectorizer = TfidfVectorizer()

In [33]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

model created using tfidf

In [34]:
nb_2 = MultinomialNB()
nb_2.fit(X_train_tfidf, y_train)

MultinomialNB()

In [35]:
nb_2_pred = nb_2.predict(X_test_tfidf)

In [36]:
print(accuracy_score(y_test, nb_2_pred))
print(confusion_matrix(y_test, nb_2_pred))
print(classification_report(y_test, nb_2_pred))

0.6609375
[[ 883    0    0    0    2   61]
 [ 145  123    0    0    2  157]
 [  48    1    9    0    1  237]
 [  41    0    0    1    3   68]
 [ 137    8    0    0   88  164]
 [  10    0    0    0    0 1011]]
              precision    recall  f1-score   support

           0       0.70      0.93      0.80       946
           1       0.93      0.29      0.44       427
           2       1.00      0.03      0.06       296
           3       1.00      0.01      0.02       113
           4       0.92      0.22      0.36       397
           5       0.60      0.99      0.74      1021

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200



In [37]:
from sklearn.linear_model import LogisticRegression

In [39]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [40]:
log_pred = log_model.predict(X_test_tfidf)

In [41]:
print(accuracy_score(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))
print(classification_report(y_test, log_pred))

0.8628125
[[893  13   1   0   7  32]
 [ 30 347   1   0  12  37]
 [ 12   5 182   0   4  93]
 [ 15   0   1  53  24  20]
 [ 24  17   1   7 302  46]
 [ 15   2  17   0   3 984]]
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       946
           1       0.90      0.81      0.86       427
           2       0.90      0.61      0.73       296
           3       0.88      0.47      0.61       113
           4       0.86      0.76      0.81       397
           5       0.81      0.96      0.88      1021

    accuracy                           0.86      3200
   macro avg       0.88      0.76      0.80      3200
weighted avg       0.87      0.86      0.86      3200

